[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YarinGaida/tabular-data-benchmarks/blob/main/supervised/supervised-deep.ipynb)

## Overview
In the `supervised-classic` notebook, we established strong baselines using traditional machine learning paradigms. We observed that while XGBoost dominates standard tabular data, high-dimensional biological data ($P \gg N$) requires extreme feature selection (like LASSO) to prevent the "Curse of Dimensionality."

In this notebook, we move to the cutting edge: **Deep Tabular Learning**. We will evaluate whether neural networks can overcome the lack of spatial inductive bias in tabular data and how they handle extreme dimensionality.

We will systematically evaluate these architectures across the exact same synthetic domains:
* **Standard Tabular Data:** A classic dataset typical of finance or standard clinical records ($N=1000, P=20$).
* **Biological Data ($P \gg N$):** A high-dimensional oncology dataset characterized by having significantly more features than samples ($N=500, P=2000$).

### Data Generation
We begin by generating our synthetic datasets, exactly as we did for the classical baselines, to ensure a fair comparison.

In [5]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from pytorch_tabular import TabularModel
from pytorch_tabular.config import DataConfig, OptimizerConfig, TrainerConfig
from pytorch_tabular.models import CategoryEmbeddingModelConfig, FTTransformerConfig
from tabpfn import TabPFNClassifier
import logging
import warnings
import os
import tabpfn_client

# Ignore standard warnings and silence excessive library logs (keeping only the essential Seed)
warnings.filterwarnings('ignore')
logging.getLogger("pytorch_tabular").setLevel(logging.WARNING)
logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)

plt.style.use('ggplot')

# Set device for GPU acceleration if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Standard Tabular Data (N=1000, P=20)
X_std, y_std = make_classification(n_samples=1000, n_features=20, n_informative=5, random_state=42)
X_train_std, X_test_std, y_train_std, y_test_std = train_test_split(X_std, y_std, test_size=0.3, random_state=42)

# Convert to Pandas DataFrames (Required for Pytorch Tabular)
train_df_std = pd.DataFrame(X_train_std, columns=[f'feature_{i}' for i in range(X_train_std.shape[1])])
train_df_std['target'] = y_train_std
test_df_std = pd.DataFrame(X_test_std, columns=[f'feature_{i}' for i in range(X_test_std.shape[1])])

# 2. High-Dimensional Biological Data (N=500, P=2000)
X_bio, y_bio = make_classification(n_samples=500, n_features=1000, n_informative=10, random_state=42)
X_train_bio, X_test_bio, y_train_bio, y_test_bio = train_test_split(X_bio, y_bio, test_size=0.3, random_state=42)

# Convert to Pandas DataFrames
train_df_bio = pd.DataFrame(X_train_bio, columns=[f'feature_{i}' for i in range(X_train_bio.shape[1])])
train_df_bio['target'] = y_train_bio
test_df_bio = pd.DataFrame(X_test_bio, columns=[f'feature_{i}' for i in range(X_test_bio.shape[1])])

print(f"Standard Data - Train: {X_train_std.shape}, Test: {X_test_std.shape}")
print(f"Biological Data - Train: {X_train_bio.shape}, Test: {X_test_bio.shape}")

Using device: cpu
Standard Data - Train: (700, 20), Test: (300, 20)
Biological Data - Train: (350, 1000), Test: (150, 1000)


## The Deep Baseline: Multi-Layer Perceptron (MLP)
We start our deep learning journey with the Multi-Layer Perceptron. While it is a universal function approximator, it lacks the built-in "greedy splitting" mechanism of decision trees.

**The Regularization Challenge:** Deep networks are notorious for memorizing noise. To combat the extreme dimensionality of our biological dataset, we must heavily rely on **Dropout** to force the network to learn robust, generalized representations rather than memorizing the training set.

In [7]:
def train_eval_pytorch_tabular(model_config, train_df, test_df, y_test):
    # Configure Data
    data_config = DataConfig(target=['target'], continuous_cols=[col for col in train_df.columns if col != 'target'], categorical_cols=[])
    
    # Hide the model architecture summary table
    trainer_config = TrainerConfig(batch_size=64, max_epochs=30, progress_bar='none', trainer_kwargs={'enable_model_summary': False})

    # Initialize model with verbose=False to silence training logs
    model = TabularModel(
        data_config=data_config, 
        model_config=model_config, 
        optimizer_config=OptimizerConfig(), 
        trainer_config=trainer_config,
        verbose=False
    )

    model.fit(train_df)

    # Evaluate
    preds = model.predict(test_df)
    return accuracy_score(y_test, preds['target_prediction'])

# --- Execution ---
mlp_config = CategoryEmbeddingModelConfig(task="classification", layers="512-256-128", dropout=0.4, activation="ReLU")

mlp_std_acc = train_eval_pytorch_tabular(mlp_config, train_df_std, test_df_std, y_test_std)
print(f"MLP Standard Accuracy: {mlp_std_acc:.3f}")

mlp_bio_acc = train_eval_pytorch_tabular(mlp_config, train_df_bio, test_df_bio, y_test_bio)
print(f"MLP Biological Accuracy: {mlp_bio_acc:.3f}")

Seed set to 42


MLP Standard Accuracy: 0.900


Seed set to 42


MLP Biological Accuracy: 0.547


In [15]:
mlp_config_2 = CategoryEmbeddingModelConfig(task="classification", layers="512-256-128", dropout=1, activation="ReLU")

mlp_std_acc = train_eval_pytorch_tabular(mlp_config_2, train_df_std, test_df_std, y_test_std)
print(f"MLP Standard Accuracy: {mlp_std_acc:.3f}")

mlp_bio_acc = train_eval_pytorch_tabular(mlp_config_2, train_df_bio, test_df_bio, y_test_bio)
print(f"MLP Biological Accuracy: {mlp_bio_acc:.3f}")

Seed set to 42


MLP Standard Accuracy: 0.517


Seed set to 42


MLP Biological Accuracy: 0.507


## State-of-the-Art Architecture: FT-Transformer
The **FT-Transformer (Feature Tokenizer + Transformer)** adapts the Attention mechanism behind Large Language Models (LLMs) for tabular data. 

**Mechanism:** It converts every numerical feature into a dense vector embedding (token). Then, Multi-Head Attention allows the network to learn complex, dynamic interactions between specific features, mimicking the way trees find complex decision boundaries.

In [ ]:
ft_config = FTTransformerConfig(
    task="classification",
    num_attn_blocks=2,
    num_heads=4,
    attn_dropout=0.2, 
    ff_dropout=0.2,   
    add_norm_dropout=0.2 
)

ft_std_acc = train_eval_pytorch_tabular(ft_config, train_df_std, test_df_std, y_test_std)
print(f"FT-Transformer Standard Accuracy: {ft_std_acc:.3f}")

# Note: Training Transformers on P=2000 is computationally heavy. We use a smaller configuration.
ft_bio_acc = train_eval_pytorch_tabular(ft_config, train_df_bio, test_df_bio, y_test_bio)
print(f"FT-Transformer Biological Accuracy: {ft_bio_acc:.3f}")

Seed set to 42


--- Training FT-Transformer ---
FT-Transformer Standard Accuracy: 0.920


Seed set to 42


FT-Transformer Biological Accuracy: 0.473


## Tabular Foundation Models: TabPFN
**TabPFN** (Tabular Prior-Data Fitted Network) represents a paradigm shift. It is a foundation model pre-trained on millions of synthetic tabular datasets.

**In-Context Learning:** TabPFN does not have a traditional training loop. You pass the training set into the model as "context," and it infers the underlying mathematical rules to predict the test set in a single forward pass.

*Constraint Note:* TabPFN is highly optimized for $N < 1000$ and $P < 100$. For our biological dataset ($P=2000$), the model typically requires prior feature selection. For this raw evaluation, we will observe its native performance limits.

### 🚀 Zero-Shot Inference with TabPFN

**Note on TabPFN Usage:**
Recent updates to the `tabpfn` library require a free API token to access the foundation model. To run the cell below locally:
1. Open https://ux.priorlabs.ai in a browser and log in (or register)
2. Accept the license on the Licenses tab
3. Copy your API token from the account page.
4. Replace `"YOUR_API_KEY_HERE"` in the code below with your actual token.

In [ ]:
# --- TabPFN License & Initialization ---
TABPFN_TOKEN = "YOUR_API_KEY_HERE" 

# Applying the token to the environment and the client
os.environ["TABPFN_TOKEN"] = TABPFN_TOKEN
tabpfn_client.set_access_token(TABPFN_TOKEN)

tabpfn_model = TabPFNClassifier(device=device) 

# --- Evaluation: Standard Data ---
tabpfn_model.fit(X_train_std, y_train_std)
preds_std = tabpfn_model.predict(X_test_std)
pfn_std_acc = accuracy_score(y_test_std, preds_std)
print(f"TabPFN Standard Accuracy: {pfn_std_acc:.3f}")

# --- Evaluation: Biological Data (Top 100 features) ---
tabpfn_model.fit(X_train_bio[:, :100], y_train_bio)
preds_bio = tabpfn_model.predict(X_test_bio[:, :100])
pfn_bio_acc = accuracy_score(y_test_bio, preds_bio)
print(f"TabPFN Biological Accuracy: {pfn_bio_acc:.3f}")

TabPFN Standard Accuracy: 0.963
TabPFN Biological Accuracy: 0.573
